# WILDFIRE PREDICT - FEATURE EXTRACTOR

This module is responsible for loading the downloaded Sentinel-2 images, and running ResNet-18 as feature extractor. The module is created as a Jupyter Notebook to have the option of running the script in `GoogleColab`, if further GPU is required for the processing of the data.

## Libraries

In [1]:
# Libraries
import os
import numpy as np
import torch
import torch.nn as nn


# from google.colab import drive
from scripts.set_parameters import PARAMETERS
from torch.utils.data import Dataset, DataLoader
from torchvision.models import (resnet18, ResNet18_Weights)

## Data Load

### Class `SentinelDataset` 

Create Class to encapsulate and easily manage the Sentinel data

In [2]:
class SentinelData(Dataset):
    def __init__(self, npz_file):
        data      = np.load(npz_file)
        self.x    = data['x']
        self.y    = data['y']
        self.keys = data['composite_key']

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        pixel_data = torch.from_numpy(self.x[idx])
        fire_label = self.y[idx]
        composite_key = str(self.keys[idx])

        return {"pixel_data": pixel_data,
                "fire_label": fire_label,
                "composite_key": composite_key}
    
    def sample_summary(self, idx=0):

        sample = self[idx]

        print("SentinelDataset Sample Summary")
        print("------------------------------")
        print("Showing full data attributes and original attributes vs (->) performed transformations\n")
        print(f"{'Total imgs':<12} : {len(self)}")
        print(f"{'Image shape':<12} : {str(self.x.shape[1:]):<20} -> {tuple(sample['pixel_data'].shape)}")
        print(f"{'Image dtype':<12} : {str(self.x.dtype):<20} -> {sample['pixel_data'].dtype}")
        print(f"{'All labels':<12} : {np.unique(self.y)}")
        print(f"{'Label':<12} : {str(self.y.dtype):<20} -> {sample['fire_label']} ({type(sample['fire_label']).__name__})")
        print(f"{'Key':<12} : {str(self.keys.dtype):<20} -> {sample['composite_key']} ({type(sample['composite_key']).__name__})")

In [3]:
test_file = PARAMETERS['PROJ_HOME']/"data"/"Sentinel2"/"2018_B001_20180101_20180117_sentinel_batch.npz"
data = SentinelData(test_file)
data.sample_summary()

SentinelDataset Sample Summary
------------------------------
Showing full data attributes and original attributes vs (->) performed transformations

Total imgs   : 787
Image shape  : (128, 128, 3)        -> (128, 128, 3)
Image dtype  : float32              -> torch.float32
All labels   : [False  True]
Label        : bool                 -> False (bool_)
Key          : int64                -> 51620180101 (str)
